In [3]:
import pandas as pd
import numpy as np

# Cargar el Dataset
df = pd.read_csv("cdmx_air_traffic_dirty.csv")

# a) Diagnóstico de valores faltantes y centinelas

# Primero veo qué columnas numéricas tengo
print("Columnas del DataFrame:")
print(df.columns)

# Cuento los NaN de cada columna numérica
print("\n--- Cantidad de NaN ---")
print("Concentracion_PM25:", df["Concentracion_PM25"].isna().sum())
print("Nivel_NO2:", df["Nivel_NO2"].isna().sum())
print("Indice_Trafico_Vehicular:", df["Indice_Trafico_Vehicular"].isna().sum())

# Cuento cuántos -999.0 hay
print("\n--- Cantidad de -999.0 ---")
print("Concentracion_PM25:", (df["Concentracion_PM25"] == -999.0).sum())
print("Nivel_NO2:", (df["Nivel_NO2"] == -999.0).sum())
print("Indice_Trafico_Vehicular:", (df["Indice_Trafico_Vehicular"] == -999.0).sum())

# Veo cuántas filas están duplicadas
print("\n--- Filas duplicadas ---")
print("Cantidad de filas duplicadas:", df.duplicated().sum())

# b) Reemplazar centinela -999.0 por np.nan y eliminar duplicados

# Reemplazo el valor centinela -999.0 por NaN (columna por columna)
df["Concentracion_PM25"] = df["Concentracion_PM25"].replace(-999.0, np.nan)
df["Nivel_NO2"] = df["Nivel_NO2"].replace(-999.0, np.nan)
df["Indice_Trafico_Vehicular"] = df["Indice_Trafico_Vehicular"].replace(-999.0, np.nan)

# Elimino las filas que están completamente duplicadas
df = df.drop_duplicates()

Columnas del DataFrame:
Index(['ID_Registro', 'Timestamp', 'Zona_Estacion', 'Concentracion_PM25',
       'Nivel_NO2', 'Indice_Trafico_Vehicular', 'Estado_Sensor'],
      dtype='str')

--- Cantidad de NaN ---
Concentracion_PM25: 1
Nivel_NO2: 0
Indice_Trafico_Vehicular: 1

--- Cantidad de -999.0 ---
Concentracion_PM25: 1
Nivel_NO2: 0
Indice_Trafico_Vehicular: 0

--- Filas duplicadas ---
Cantidad de filas duplicadas: 1


In [ ]:
from sklearn.impute import KNNImputer
import pandas as pd
import numpy as np

df_clean = df.copy()   

# 1. Homogeneizar 'Zona_Estacion'
# Ver cómo están escritos los valores
print("Valores únicos de Zona_Estacion antes:")
print(df_clean["Zona_Estacion"].unique())

# Limpiar textos
df_clean["Zona_Estacion"] = df_clean["Zona_Estacion"].str.strip()   
df_clean["Zona_Estacion"] = df_clean["Zona_Estacion"].str.replace(" ", "_")
df_clean["Zona_Estacion"] = df_clean["Zona_Estacion"].str.title()

# Dejar solo las categorías limpias que me piden
df_clean["Zona_Estacion"] = df_clean["Zona_Estacion"].replace({
    "Centro_Historico": "Centro_Historico",
    "Centro Historico": "Centro_Historico",
    "Centro_historico": "Centro_Historico",
    "CENTRO_HISTORICO": "Centro_Historico",
    "Zona_Oriente": "Zona_Oriente",
    "Zona Oriente": "Zona_Oriente",
    "Zona_Sur": "Zona_Sur",
    "Zona Sur": "Zona_Sur"
})

print("\nValores únicos de Zona_Estacion después:")
print(df_clean["Zona_Estacion"].unique())

# 2. Convertir Timestamp a datetime y extraer Hora_Dia

# Convertir la columna Timestamp a tipo fecha/hora
df_clean["Timestamp"] = pd.to_datetime(df_clean["Timestamp"])

# Extraer solo la hora del día (0 a 23)
df_clean["Hora_Dia"] = df_clean["Timestamp"].dt.hour

print("\nColumna Hora_Dia creada:")
print(df_clean[["Timestamp", "Hora_Dia"]].head()) 

# 3. Detección de outliers por IQR en 'Concentracion_PM25' y sustitución por np.nan
 
# Calcular el rango intercuartílico (IQR)
Q1 = df_clean["Concentracion_PM25"].quantile(0.25)
Q3 = df_clean["Concentracion_PM25"].quantile(0.75)
IQR = Q3 - Q1

# Límites de detección de outliers
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"\nLímite inferior IQR: {limite_inferior}")
print(f"Límite superior IQR: {limite_superior}")

# Reemplazar los valores que están fuera de los límites por NaN
df_clean.loc[df_clean["Concentracion_PM25"] > limite_superior, "Concentracion_PM25"] = np.nan
df_clean.loc[df_clean["Concentracion_PM25"] < limite_inferior, "Concentracion_PM25"] = np.nan

print("\nValores de Concentracion_PM25 después de quitar outliers:")
print(df_clean["Concentracion_PM25"]) 
 
# 4. Imputación avanzada k-NN sobre variables numéricas
 
# Columnas que se van imputar
columnas_numericas = ["Concentracion_PM25", "Nivel_NO2", "Indice_Trafico_Vehicular"]

# Crear el imputador
imputer = KNNImputer(n_neighbors=2)

# Aplicar la imputación
df_clean[columnas_numericas] = imputer.fit_transform(df_clean[columnas_numericas])





Valores únicos de Zona_Estacion antes:
<ArrowStringArray>
['Centro Historico', 'centro historico', 'CENTRO HISTORICO',
     'Zona Oriente',     'zona oriente',         'Zona Sur']
Length: 6, dtype: str

Valores únicos de Zona_Estacion después:
<ArrowStringArray>
['Centro_Historico', 'Zona_Oriente', 'Zona_Sur']
Length: 3, dtype: str

Columna Hora_Dia creada:
            Timestamp  Hora_Dia
0 2026-10-01 08:00:00         8
1 2026-10-01 09:00:00         9
2 2026-10-01 10:00:00        10
3 2026-10-01 11:00:00        11
4 2026-10-01 12:00:00        12

Límite inferior IQR: -44.61250000000001
Límite superior IQR: 125.48750000000001

Valores de Concentracion_PM25 después de quitar outliers:
0     35.4
1     42.1
2      NaN
3    120.5
4      NaN
5      NaN
6     18.2
7    -20.1
8     19.5
9     19.5
Name: Concentracion_PM25, dtype: float64


In [7]:



# 1. Estadísticas de forma para Concentracion_PM25

print("--- Estadisticas de Concentracion_PM25 ---")

media = df_clean["Concentracion_PM25"].mean()
mediana = df_clean["Concentracion_PM25"].median()
desv_estandar = df_clean["Concentracion_PM25"].std()
asimetria = df_clean["Concentracion_PM25"].skew()
curtosis = df_clean["Concentracion_PM25"].kurt()

print(f"Media: {media}")
print(f"Mediana: {mediana}")
print(f"Desviacion estándar: {desv_estandar}")
print(f"Skewness (asimetría): {asimetria}")
print(f"Kurtosis (curtosis): {curtosis}")

# 2. Matrices de correlación Pearson y Spearman

# Solo las columnas numéricas
columnas_numericas = ["Concentracion_PM25", "Nivel_NO2", "Indice_Trafico_Vehicular", "Hora_Dia"]

print("\n--- Matriz de correlacion de Pearson ---")
matriz_pearson = df_clean[columnas_numericas].corr(method="pearson")
print(matriz_pearson)

print("\n--- Matriz de correlacion de Spearman ---")
matriz_spearman = df_clean[columnas_numericas].corr(method="spearman")
print(matriz_spearman)

# 3. Pregunta diagnóstica: ¿hay multicolinealidad?

print("\n--- Diagnostico de multicolinealidad ---")
print("Buscamos pares de variables con |r| >= 0.85")


--- Estadisticas de Concentracion_PM25 ---
Media: 39.055
Mediana: 37.075
Desviacion estándar: 37.81235431560549
Skewness (asimetría): 0.945501369423778
Kurtosis (curtosis): 1.9615575771525489

--- Matriz de correlacion de Pearson ---
                          Concentracion_PM25  Nivel_NO2  \
Concentracion_PM25                  1.000000   0.627559   
Nivel_NO2                           0.627559   1.000000   
Indice_Trafico_Vehicular            0.317164   0.206906   
Hora_Dia                           -0.418914  -0.303540   

                          Indice_Trafico_Vehicular  Hora_Dia  
Concentracion_PM25                        0.317164 -0.418914  
Nivel_NO2                                 0.206906 -0.303540  
Indice_Trafico_Vehicular                  1.000000 -0.749666  
Hora_Dia                                 -0.749666  1.000000  

--- Matriz de correlacion de Spearman ---
                          Concentracion_PM25  Nivel_NO2  \
Concentracion_PM25                  1.000000   0.9480